# nl2sh QLoRA training (Colab T4 / Kaggle) - self-contained, fixed

**How to use:** Runtime > Change runtime type > **T4 GPU**. Then **Runtime > Run all**.
After cell 2 finishes, if Colab shows "Restart session", click it, then run from cell 3 onward (do NOT re-run cell 2).

This notebook does **not** clone any GitHub repo and does **not** need `requirements.txt` or `train_qlora.py`. Everything is inside the notebook, so the old `CalledProcessError (exit status 1)` and dependency-conflict errors are gone.

In [ ]:
# Cell 1 - check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU.  Kaggle: Settings > Accelerator > GPU T4.")
props = torch.cuda.get_device_properties(0)
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM:", round(props.total_memory / 1024**3, 1), "GiB")
print("torch:", torch.__version__)

In [ ]:
# Cell 2 - install libraries (do NOT reinstall torch/numpy - that breaks Colab)
!pip install -q -U transformers peft accelerate bitsandbytes datasets
import importlib.metadata as md
for p in ["transformers", "peft", "accelerate", "bitsandbytes", "datasets"]:
    print(p, md.version(p))

In [ ]:
# Cell 3 - settings (edit here only)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

BASE_MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"   # small + fast on free T4. Bigger option: "Qwen/Qwen2.5-1.5B-Instruct"
DATASET      = "westenfelder/NL2SH-ALFA"
TRAIN_ROWS   = 10000     # quick run (~30-60 min on T4). Set to None to use the full train split (several hours)
MAX_LEN      = 256
EPOCHS       = 1
BATCH        = 8
GRAD_ACCUM   = 2         # effective batch = 16
LR           = 2e-4
LORA_R       = 16
SEED         = 42

OUT_DIR = "/content/nl2sh-training" if os.path.exists("/content") else "/kaggle/working/nl2sh-training"
os.makedirs(OUT_DIR, exist_ok=True)
print("Output folder:", OUT_DIR)

In [ ]:
# Cell 4 - (optional) save checkpoints to Google Drive so a runtime reset does not lose them
# Skip this cell if you don't want Drive. Colab only.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/nl2sh-training"
    os.makedirs(OUT_DIR, exist_ok=True)
    print("Using Drive folder:", OUT_DIR)
except Exception as e:
    print("Drive not used (fine):", e)

In [ ]:
# Cell 5 - load + prepare dataset
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

ds = load_dataset(DATASET, "train", split="train")
print(ds)
print("Columns:", ds.column_names)

# find the natural-language column and the bash column
cols = ds.column_names
NL_COL   = next(c for c in ["nl", "instruction", "prompt", "query", "question"] if c in cols)
BASH_COL = next(c for c in ["bash", "command", "output", "response", "answer"] if c in cols)
print("Using:", NL_COL, "->", BASH_COL)

ds = ds.filter(lambda r: r[NL_COL] and r[BASH_COL])
ds = ds.shuffle(seed=SEED)
if TRAIN_ROWS:
    ds = ds.select(range(min(TRAIN_ROWS, len(ds))))
print("Training rows:", len(ds))

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

SYSTEM = "Translate the user's request into a single Bash command. Output only the command."

def build_prompt(nl):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": nl}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def tokenize(row):
    prompt = build_prompt(row[NL_COL].strip())
    answer = row[BASH_COL].strip() + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]   # loss only on the command
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

tokenized = ds.map(tokenize, remove_columns=ds.column_names, num_proc=1)
tokenized = tokenized.filter(lambda r: any(l != -100 for l in r["labels"]))
print("After tokenizing:", len(tokenized))
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

In [ ]:
# Cell 6 - load 4-bit model + LoRA
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16   # T4 has no bf16 -> fp16

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map={"": 0}, torch_dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_R * 2, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Cell 7 - train (resumes automatically from the newest checkpoint if one exists)
import glob, json, os
from transformers import Trainer, TrainingArguments, set_seed
set_seed(SEED)

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=not use_bf16,
    bf16=use_bf16,
    optim="adamw_torch",
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    dataloader_num_workers=0,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)

has_ckpt = bool(glob.glob(os.path.join(OUT_DIR, "checkpoint-*")))
trainer.train(resume_from_checkpoint=True if has_ckpt else None)

final_dir = os.path.join(OUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_dir)
tok.save_pretrained(final_dir)
with open(os.path.join(final_dir, "training_metadata.json"), "w") as f:
    json.dump({"base_model": BASE_MODEL, "dataset": DATASET, "rows": len(tokenized), "epochs": EPOCHS,
               "max_len": MAX_LEN, "lora_r": LORA_R, "lr": LR, "seed": SEED}, f, indent=2)
print("Saved adapter to:", final_dir)

In [ ]:
# Cell 8 - quick test of the trained model
model.eval()
model.config.use_cache = True
tests = ["list all python files in the current directory",
         "show disk usage of each folder sorted by size",
         "find files larger than 100MB",
         "count lines in all .txt files"]
for t in tests:
    inputs = tok(build_prompt(t), return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tok.pad_token_id)
    print(f"> {t}\n  {tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()}\n")

In [ ]:
# Cell 9 - zip the adapter and download it
import shutil, os
final_dir = os.path.join(OUT_DIR, "final_adapter")
assert os.path.exists(os.path.join(final_dir, "adapter_config.json")), "Adapter missing - training did not finish."
zip_base = "/content/nl2sh-qlora-adapter" if os.path.exists("/content") else "/kaggle/working/nl2sh-qlora-adapter"
zip_path = shutil.make_archive(zip_base, "zip", final_dir)
print("ZIP:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download it from the file browser on the left.")

## Notes
- **Out of memory?** Lower `BATCH` to 4 (and raise `GRAD_ACCUM` to 4), or `MAX_LEN` to 192.
- **Runtime disconnected?** Re-run cells 1, 2, 3, (4), 5, 6, 7. Cell 7 resumes from the latest checkpoint (works best with the Drive cell).
- **Want the full dataset?** Set `TRAIN_ROWS = None` in cell 3. It takes several hours on free T4.
- The GGUF export step was removed on purpose: it needs a C++ build of llama.cpp and was the main cause of the tar / exit-status errors. Do it after training, as a separate step.